### Computing mAP using the COCO evaluator for a single detection model

In [8]:
import json
import csv
from mAP_utils import *

In [9]:
import degirum as dg, mytools
cloud_token = mytools.get_token() # get cloud API access token from env.ini file
cloud_zoo_url = mytools.get_cloud_zoo_url() # get cloud zoo URL from env.ini file
zoo = dg.connect(dg.CLOUD, cloud_zoo_url, cloud_token)

In [10]:
# zoo.list_models()

You need to have two json files in the same directory:

* `INPUT_DATA.json` : You need to specify the image_folder_path (local path) on which the evaluation is done and the ground_truth_annotations path (loacl path) for the labels for each of the classes. There has to be two keys in this json, one is the "image_folder_path" and the other "ground_truth_annotations_path".The "image_folder_path" key should have the different categories like 'coco' ,'hand', 'face', 'lp' , 'car' and its corresponding image data path. The "ground_truth_annotations_path" key should the labels file path (labels.json) for each category.
   
   {
    
    "image_folder_path" :
    
        {
            "car" : "./car_coco_val/car_20K_coco_val/data/",
            
            "hand" : "./human_hand_coco_val/data/",
            
            "face" : "./face_det/val/data/",
            
            "coco" : "./coco_val/coco_images/",
            
            "lp" : "./lp_det/val/data/"
            
        },
        
    "ground_truth_annotations_path" : 
    
        {
            "car" : "./car_coco_val/car_20K_coco_val/labels.json",
            
            "hand" : "./human_hand_coco_val/labels.json",
            
            "face" : "./face_det/val/labels.json",
            
            "coco" : "./coco_val/labels.json",
            
            "lp" : "./lp_det/val/labels.json"

        }

    }
    

* `CLASSMAP_CONFIG.json` : You need to specify the classmaps for each category.

    For example, this is how the json should look like : 

         {

            "ClassMap" : 
            { 
               "coco" : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 31, 32, 33,34,35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63,64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90],      
                "car" : [0],  
                "hand" : [0],
                "face" : [1],
                "lp" : [0]

            }

        }


In [11]:
model_name = "yolov5nu_relu6_lp--640x640_quant_n2x_orca_1"

with open('./coco_configs/classmap_config.json') as config:
    classmap_categories=json.load(config)
               
with open('./coco_configs/input_data.json') as df:
    input_json = json.load(df)
       
pred_path = model_name + "_predictions.json"

### Input json parameters 

In [12]:
if "coco" in model_name:
    image_folder_path = input_json["image_folder_path"]["coco"]
    ground_truth_annotations_path = input_json["ground_truth_annotations_path"]["coco"]
    class_map = classmap_categories["ClassMap"]["coco"]

elif "car" in model_name:
    image_folder_path = input_json["image_folder_path"]["car"]
    ground_truth_annotations_path = input_json["ground_truth_annotations_path"]["car"]
    class_map = classmap_categories["ClassMap"]["car"]

elif "hand" in model_name:
    image_folder_path = input_json["image_folder_path"]["hand"]
    ground_truth_annotations_path = input_json["ground_truth_annotations_path"]["hand"]
    class_map = classmap_categories["ClassMap"]["hand"]

elif "face" in model_name:
    image_folder_path = input_json["image_folder_path"]["face"]
    ground_truth_annotations_path = input_json["ground_truth_annotations_path"]["face"]
    class_map = classmap_categories["ClassMap"]["face"]

elif "lp" in model_name:
    image_folder_path = input_json["image_folder_path"]["lp"]
    ground_truth_annotations_path = input_json["ground_truth_annotations_path"]["lp"]
    class_map = classmap_categories["ClassMap"]["lp"]

In [13]:
# input_postprocess_parameters = {"OutputConfThreshold" : 0.001  ,"OutputNMSThreshold" : 0.6, "MaxDetections" : 100 , "MaxDetectionsPerClass" : 20, "MaxClassesPerDetection" : 1, "UseRegularNMS" : True, "InputResizeMethod" : "bilinear" , "InputPadMethod" : "letterbox", "ImageBackend" : "opencv", "InputImgFmt" : "JPEG"}

### Load the eval model with the postprocess parameters and run the coco evaluation

In [14]:
model=zoo.load_model(model_name)
map_evaluator=ObjectDetectionModelEvaluator(model,image_folder_path,ground_truth_annotations_path,class_map,pred_path,print_frequency=50)
map_results=map_evaluator.evaluate()

loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


100%|████████████████████████████████████████████████████████████████████████████| 724/724 [00:00<00:00, 616709.20it/s]


DegirumException: Failed to perform model 'degirum/ultralytics_v3/yolov5nu_relu6_lp--640x640_quant_n2x_orca_1' inference: Timeout waiting for space in frame queue during inference of model 'degirum/ultralytics_v3/yolov5nu_relu6_lp--640x640_quant_n2x_orca_1' (timeout = 30.0 s, queue depth = 80)

### Saving the results to a csv file

In [ ]:
save_mAP_path = './mAP_results1.csv'

In [ ]:
header = ['model_name','mAP50', 'mAP75', 'mAP0.50:0.95', 'mAP0.50-0.95','mAP0.50-0.95','mAP0.50-0.95','mAP0.50-0.95','mAP0.50-0.95','mAP0.50-0.95', 'mAP0.50-0.95','mAP0.50-0.95']
data = [eval_stats.tolist()]
data[0].insert(0,model_name)
with open(save_mAP_path, 'w', encoding='UTF8', newline ='') as f:
    writer = csv.writer(f, delimiter=',') 
    writer.writerow(header)
    writer.writerows(data)
    f.close()